# AI Translation Matcher – Textual Similarity Only
This notebook evaluates the retrieval of similar documents using text content only, based on TF-IDF vectorization and cosine similarity (via `NearestNeighbors`). It uses the precomputed index stored in `output/text_vectors.pkl`.

It is designed to test the speed and quality of text-based search, including OCR-extracted text.

In [ ]:
import sys
import os
sys.path.append(os.path.abspath("../../src"))

import pickle
import pandas as pd
import numpy as np
import time
from model import extract_text
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import normalize
from PIL import Image, UnidentifiedImageError
import fitz
from docx import Document
import io
import ipywidgets as widgets
from IPython.display import display as ipy_display
from fetch import download_file_from_space
from IPython.display import Image as IPyImage, display

In [ ]:
test_dir = "../../AI_matcher/tests"

available_files = sorted([
    f for f in os.listdir(test_dir)
    if os.path.isfile(os.path.join(test_dir, f)) and not f.startswith(".DS_Store")
])

dropdown = widgets.Dropdown(
    options=available_files,
    description="File:",
    disabled=False
)

ipy_display(dropdown)

In [ ]:
def display_document_preview(file_key: str):
    """
    Display a preview of any document (image, PDF, DOCX) using IPython rendering.
    """
    print(f"\nPreview: {file_key}")
    try:
        content = download_file_from_space(file_key)

        if file_key.lower().endswith((".jpg", ".jpeg", ".png")):
            img = Image.open(io.BytesIO(content))
            display(img)

        elif file_key.lower().endswith(".pdf"):
            try:
                doc = fitz.open(stream=content, filetype="pdf")
                pix = doc[0].get_pixmap(dpi=150)
                png_bytes = pix.tobytes("png")
                ipy_img = IPyImage(data=png_bytes)
                display(ipy_img)
            except Exception as e:
                print(f"[ERROR] Could not preview PDF {file_key}: {e}")

        elif file_key.lower().endswith(".docx"):
            doc = Document(io.BytesIO(content))
            for rel in doc.part._rels:
                rel_obj = doc.part._rels[rel]
                if "image" in rel_obj.target_ref:
                    img_data = rel_obj.target_part.blob
                    img = Image.open(io.BytesIO(img_data))
                    display(img)
                    break
            else:
                print("[INFO] No image found in the .docx file.")

        else:
            print("[SKIP] Unsupported file type for preview.")

    except Exception as e:
        print(f"[ERROR] Could not display {file_key}: {e}")

In [ ]:
def run_similarity_analysis():
    query_path = os.path.join(test_dir, dropdown.value)
    tfidf_path = "../../output/text_vectors.pkl"
    top_k = 5

    with open(tfidf_path, "rb") as f:
        data = pickle.load(f)

    keys = data["keys"]
    matrix = data["vectors"]
    vectorizer = data["vectorizer"]

    with open(query_path, "rb") as f:
        content = f.read()
    query_text = extract_text(content, query_path)

    print("\nExtracted text (first 300 characters):")
    print(query_text[:300])

    query_vec = vectorizer.transform([query_text])
    query_vec = normalize(query_vec)

    start = time.time()
    nn = NearestNeighbors(n_neighbors=top_k, metric="cosine")
    nn.fit(matrix)
    distances, indices = nn.kneighbors(query_vec)
    end = time.time()

    results = [(keys[i], round((1 - distances[0][j]) * 100, 2)) for j, i in enumerate(indices[0])]
    df = pd.DataFrame(results, columns=["file", "similarity_%"])

    print("\nTop text-based matches via TF-IDF + KNN:")
    print(df)

    print("\nDisplaying matched document previews:")
    for file_key in df["file"]:
        display_document_preview(file_key)

    print(f"\nExecution time: {end - start:.2f} seconds")

In [ ]:
run_similarity_analysis()